# Level 1: Junior Foundations
**Course:** DBX Study Guide | **Duration:** ~2 hours | **Next:** `11_junior_test`

## Learning objectives
By the end of this class you will be able to:
1. Use a repeatable 5-step method to solve any coding question out loud
2. Pick a **hash map or set** to turn O(n^2) work into O(n)
3. Solve array problems in **one pass** with running state and two pointers
4. Write **binary search** without off-by-one bugs
5. Traverse a graph with **BFS**
6. Explain Spark's **driver/executor** model, **lazy evaluation**, and the main **join types**
7. Use the core vocabulary of **data warehousing**: fact, dimension, grain, SCD1/SCD2, medallion

## How this class works
| Marker | What to do |
|---|---|
| **CONCEPT** | Read it. The idea comes before the code. |
| **DEMO** | Run the cell, then read the **Breakdown** under it. |
| **KNOWLEDGE CHECK** | Say or write your prediction *before* running the next cell. |
| **YOUR TURN** | Write the code. The cell checks itself. Use `lesson.hint("id")` if you're stuck, or `lesson.solution("id")` to see the answer. |

> **Tip:** open the **Table of contents** (the list icon in the left sidebar) to jump between lessons. Run cells **top to bottom**,
> using Serverless compute.

## Classroom setup
Run the next cell first. It loads the course libraries and the exercise checker (`lesson`).

In [ ]:
%run ./Includes/Classroom-Setup

### Extra imports for this class
`time` for quick timings, `Counter` for counting, and `deque` for BFS queues.

In [ ]:
import time
from collections import Counter, deque

# Lesson 1: How to answer any coding question

### CONCEPT: The 5-step method
Interviewers grade **how you think** as much as the final code. Use the same 5 steps every time, out loud:

| Step | What you say | Why it matters |
|---|---|---|
| 1. Clarify | "Can the input be empty? Sorted? Duplicates? Negatives? What do I return if there's no answer?" | Avoids solving the wrong problem |
| 2. Brute force | "The simple way is ... and it's O(n^2)." | You always have *a* working answer |
| 3. Optimize | "The repeated work is ..., so I'll use a ..." | This is where the points are |
| 4. Code + trace | Write it, then walk through one small example | Catches bugs before the interviewer does |
| 5. Complexity | "O(n) time, O(n) space, and the edge cases are ..." | Shows you understand the trade-off |

We'll apply it to the most famous interview question: **Two Sum**.

### Step 1: Clarify
> *Given a list `nums` and a `target`, return the indices of two numbers that add up to `target`.*

Good clarifying questions: *Is there always exactly one answer? Can I use the same element twice? Is the list sorted?*
Assume: exactly one answer, the same index can't be used twice, and the list is **not** sorted.

### Step 2: Brute force (DEMO)
Try every pair.

In [ ]:
def two_sum_brute(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
    return []

two_sum_brute([2, 7, 11, 15], 9)

**Breakdown**
- The outer loop runs n times, and the inner loop runs up to n times, so there are about **n^2 / 2 comparisons**: **O(n^2)** time.
- It uses no extra memory: **O(1)** space.
- The repeated work: for every `nums[i]` we *search* the rest of the list for `target - nums[i]`. **Searching is the expensive part.**

### Step 3: Optimize
If searching is the slow part, use a structure where searching is **O(1)**: a **dictionary** (hash map).
As we walk the list, we remember `value -> index` for everything seen so far. For each new number we ask:
*"Have I already seen the number I need?"*

In [ ]:
def two_sum(nums, target):
    seen = {}                       # value -> index
    for i, x in enumerate(nums):
        need = target - x
        if need in seen:            # O(1) average lookup
            return [seen[need], i]
        seen[x] = i                 # store AFTER checking, so x is never paired with itself
    return []

two_sum([2, 7, 11, 15], 9)

### Step 4: Trace by hand
`two_sum([3, 2, 4], 6)`:

| i | x | need | `seen` before this step | action |
|---|---|---|---|---|
| 0 | 3 | 3 | `{}` | 3 not seen, so store `{3: 0}` |
| 1 | 2 | 4 | `{3: 0}` | 4 not seen, so store `{3: 0, 2: 1}` |
| 2 | 4 | 2 | `{3: 0, 2: 1}` | **2 seen at index 1**, so return `[1, 2]` |

Notice that index 0 (value 3) is never paired with itself, even though 3 + 3 = 6. That's why we store *after* checking.

### Step 5: Complexity
**O(n) time** (one pass, O(1) lookups) and **O(n) space** (the dict). We traded memory for speed, which is the most common trade-off in interviews.

### KNOWLEDGE CHECK
For n = 4,000 numbers with **no** answer (the worst case), roughly how many times slower is the brute force? Make a guess, then run the next cell.

In [ ]:
nums = list(range(4000))
for f in (two_sum_brute, two_sum):
    t0 = time.perf_counter()
    f(nums, -1)
    print(f"{f.__name__:<14} {time.perf_counter() - t0:.4f}s")

**Answer:** usually **hundreds of times slower**. Brute force does about 8,000,000 comparisons, while the dict does 4,000 lookups.
If you double n, the brute force gets **4x** slower but the dict version only **2x**. That's what O(n^2) vs O(n) *means*.

### YOUR TURN (J-1): First repeated value
Return the **first value that appears for the second time** as you scan left to right, or `None` if nothing repeats.
`[2, 5, 1, 2, 3, 5]` returns `2` (2 repeats at index 3, before 5 repeats at index 5). Target: **O(n)**.

In [ ]:
def first_repeated(nums):
    # TODO: your code here
    raise NotImplementedError

lesson.check("J-1", first_repeated)

# Lesson 2: Hash maps, sets, and strings

### CONCEPT: Why hash lookups are O(1)
A hash map runs `hash(key)` to compute **which bucket** the key lives in, then looks in that bucket. It doesn't scan everything.
- `set`: stores only keys. Use it to answer "have I seen x?"
- `dict`: stores key -> value pairs. Use it for "where did I see x?" or "how many times?"
- `Counter`: a dict built for counting.

**Spark connection:** a shuffle sends each row to partition `hash(key) % num_partitions`. It's the same idea, spread across machines.

### DEMO: list vs set membership

In [ ]:
data_list = list(range(200_000))
data_set = set(data_list)
lookups = range(199_000, 200_000)           # 1,000 lookups near the end of the list

t0 = time.perf_counter(); hits = sum(1 for x in lookups if x in data_list); t_list = time.perf_counter() - t0
t0 = time.perf_counter(); hits = sum(1 for x in lookups if x in data_set);  t_set = time.perf_counter() - t0
print(f"list: {t_list:.4f}s   set: {t_set:.6f}s   ({t_list / max(t_set, 1e-9):,.0f}x faster)")

**Breakdown:** `x in list` checks elements one by one, which is O(n) per lookup. `x in set` jumps straight to one bucket, which is O(1) on average.
The rule of thumb: **if you write `in` inside a loop, the right-hand side should almost always be a set or dict.**

### DEMO: Counting with `Counter`

In [ ]:
c = Counter("mississippi")
print(c)
print("most common:", c.most_common(2))
print("count of 'z':", c["z"])          # a missing key returns 0, no KeyError

### CONCEPT: Valid anagram, two ways
Two strings are anagrams if they contain **the same letters with the same counts**.
- **Sort both and compare**: O(n log n).
- **Count both and compare**: O(n). Better, and it's the pattern behind `groupBy().count()`.

In [ ]:
def is_anagram(a, b):
    if len(a) != len(b):          # cheap early exit
        return False
    return Counter(a) == Counter(b)

print(is_anagram("listen", "silent"), is_anagram("rat", "car"))

### KNOWLEDGE CHECK
What does `is_anagram("aacc", "ccac")` return? Count the letters yourself first.

In [ ]:
print(Counter("aacc"), Counter("ccac"), is_anagram("aacc", "ccac"))

**Answer:** `False`. `aacc` has `a:2, c:2` while `ccac` has `c:3, a:1`. They have the same length and the same *set* of letters but different **counts**.
This is why `set(a) == set(b)` is a wrong solution.

### YOUR TURN (J-2): First unique character
Return the **index** of the first character that appears exactly once in `s`, or `-1`. `"loveleetcode"` returns `2` (`v`).

In [ ]:
def first_unique_char(s):
    # TODO
    raise NotImplementedError

lesson.check("J-2", first_unique_char)

# Lesson 3: Arrays and one-pass thinking

### CONCEPT: Carry running state instead of looking back
Many array problems seem to need "look at every earlier element", which is O(n^2). Ask yourself:
**what single value about the past do I actually need?** Keep that one value updated as you go.

> *Best time to buy and sell:* buy once, sell later, and maximize profit. To sell today, you only need **the lowest price so far**.

### DEMO: max profit, with a trace

In [ ]:
def max_profit(prices, trace=False):
    lowest, best = float("inf"), 0
    for day, p in enumerate(prices):
        lowest = min(lowest, p)
        best = max(best, p - lowest)
        if trace:
            print(f"day {day}: price={p:<3} lowest_so_far={lowest:<3} best_profit={best}")
    return best

max_profit([7, 1, 5, 3, 6, 4], trace=True)

**Breakdown**
- On day 1 the price is 1, so `lowest` becomes 1. Every later day measures profit against 1.
- On day 4 the price is 6, giving a profit of 5, which is the best.
- We never compared day 4 with days 0, 2 or 3 directly. The running `lowest` summarized them. That makes it **O(n) time and O(1) space**.

### CONCEPT: Two pointers
Put one pointer at each end and move them toward each other. Each step discards at least one element, so the whole scan is O(n).
Classic use: palindromes, and pair sums in a **sorted** list.

In [ ]:
def is_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if not s[i].isalnum():
            i += 1
        elif not s[j].isalnum():
            j -= 1
        elif s[i].lower() != s[j].lower():
            return False
        else:
            i, j = i + 1, j - 1
    return True

print(is_palindrome("A man, a plan, a canal: Panama"), is_palindrome("race a car"))

**Breakdown:** skipping non-letters *inside* the loop avoids building a cleaned copy of the string, which saves O(n) extra memory.
Mention this trade-off; interviewers like it.

### YOUR TURN (J-3): Longest run of 1s
Given a list of 0s and 1s, return the length of the longest run of consecutive 1s. `[1,1,0,1,1,1]` returns `3`. One pass, O(1) space.

In [ ]:
def max_consecutive_ones(nums):
    # TODO
    raise NotImplementedError

lesson.check("J-3", max_consecutive_ones)

# Lesson 4: Binary search

### CONCEPT: Halve the search space
Binary search only works on **sorted** data. Look at the middle element. If it's too small, the answer must be to the right, so discard the left half.
Each step halves the range, giving **O(log n)**: 1,000,000 items take about 20 steps, and 1,000,000,000 take about 30.

**The invariant (say this out loud):** *"If the target exists, it's always inside `nums[lo..hi]`."* Every update must keep that true.

### DEMO: watch lo / mid / hi move

In [ ]:
def binary_search(nums, target, trace=False):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if trace:
            print(f"lo={lo:<3} hi={hi:<3} mid={mid:<3} nums[mid]={nums[mid]}")
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1          # mid is too small, and so is everything left of it
        else:
            hi = mid - 1          # mid is too big, and so is everything right of it
    return -1

nums = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print("index:", binary_search(nums, 23, trace=True))

**Breakdown:** we started with 10 candidates, then 5, then 2, then found it. We used `mid + 1` / `mid - 1` because `mid` itself was already checked.

### The three classic bugs
| Bug | Symptom |
|---|---|
| `while lo < hi` with an inclusive `hi` | misses the target when only one candidate is left |
| `lo = mid` instead of `mid + 1` | **infinite loop** when `lo` and `hi` are next to each other |
| searching unsorted data | silently wrong answers |

### KNOWLEDGE CHECK
`binary_search([5, 1, 4, 2], 5)`: the list is **not sorted**. Is 5 found? Trace it in your head.

In [ ]:
print(binary_search([5, 1, 4, 2], 5, trace=True))

**Answer:** it returns `-1` even though 5 is at index 0. At mid=1 it sees 1 < 5 and throws away the left half, which held the 5.
**Always confirm the input is sorted** (or sort it first: O(n log n) once, then each search is O(log n)).

### YOUR TURN (J-4): Search insert position
`nums` is sorted. Return the index of `target` if it's present, otherwise the index where it **would be inserted** to keep the list sorted.
`([1,3,5,6], 2)` returns `1`, and `([1,3,5,6], 7)` returns `4`. Target: O(log n). Don't use `bisect`.

In [ ]:
def search_insert(nums, target):
    # TODO
    raise NotImplementedError

lesson.check("J-4", search_insert)

# Lesson 5: Graphs and BFS

### CONCEPT: Representing a graph
Use an **adjacency list**: a dict from each node to the list of nodes it points to. It's compact (O(V + E)) and makes "who are my neighbors?" O(1).

A data pipeline is a graph:
```
raw ──> stg_orders ─────────────┐
 └────> stg_customers ──> dim_customer ──> fact_sales ──> dashboard
```

In [ ]:
edges = [("raw", "stg_orders"), ("raw", "stg_customers"), ("stg_customers", "dim_customer"),
         ("stg_orders", "fact_sales"), ("dim_customer", "fact_sales"), ("fact_sales", "dashboard")]
graph = {}
for src, dst in edges:
    graph.setdefault(src, []).append(dst)
    graph.setdefault(dst, [])
graph

### CONCEPT: Breadth-first search
BFS explores in **rings**: everything 1 hop away, then 2 hops, and so on. That's why it finds the **fewest-hops** path.
- Use a **queue** (`deque`): first in, first out.
- Mark a node visited **when you add it to the queue**, not when you pop it. Otherwise it can be queued twice.

In [ ]:
def bfs_levels(graph, start, trace=False):
    dist = {start: 0}               # doubles as the visited set
    q = deque([start])
    while q:
        node = q.popleft()          # deque.popleft is O(1); list.pop(0) is O(n)
        for nb in graph.get(node, []):
            if nb not in dist:
                dist[nb] = dist[node] + 1
                q.append(nb)
        if trace:
            print(f"visited {node:<14} queue now: {list(q)}")
    return dist

bfs_levels(graph, "raw", trace=True)

**Breakdown:** `fact_sales` was first reached from `stg_orders` at distance 2. When `dim_customer` tried to add it again (that path is 3 hops), it was already
in `dist`, so it was skipped. **Complexity: O(V + E)**, since every node is queued once and every edge is checked once.

### KNOWLEDGE CHECK
What happens on a graph with a **cycle**, `{1: [2], 2: [3], 3: [1]}`, if we *don't* keep a visited set?

**Answer:** it loops forever (1, 2, 3, 1, 2, ...). The visited set is what guarantees termination.
Real-world version: Unity Catalog lineage graphs and job dependency graphs can contain unexpected cycles, so always track visited nodes.

### YOUR TURN (J-5): Reachable nodes
Return the **set** of all nodes reachable from `start`, including `start`. BFS or DFS both work. It must handle cycles.

In [ ]:
def reachable(graph, start):
    # TODO
    raise NotImplementedError

lesson.check("J-5", reachable)

# Lesson 6: PySpark fundamentals

### CONCEPT: Driver and executors
```
           ┌──────────── DRIVER (your notebook) ────────────┐
           │ builds the plan, schedules tasks, gets results │
           └───────────────┬────────────────────────────────┘
       ┌───────────────────┼────────────────────┐
  EXECUTOR 1          EXECUTOR 2           EXECUTOR 3
  partitions 1-4      partitions 5-8       partitions 9-12   <- the data lives and is processed here
```
A DataFrame is split into **partitions**, which are processed in parallel by executors. The **driver** is one machine. Anything you pull
back to it (`collect()`, `toPandas()`) must fit in its memory.

In [ ]:
d = lesson.data("junior")
orders, customers = d["orders"], d["customers"]
display(orders)

And here's the `customers` table: 6 customers, 2 of whom (Eli and Fay) have no orders. That matters in the join lesson.

In [ ]:
display(customers)

### CONCEPT: Transformations vs actions (lazy evaluation)
| Transformations: *build the plan* | Actions: *run the plan* |
|---|---|
| `select`, `filter`/`where`, `withColumn`, `join`, `groupBy().agg()`, `orderBy` | `count`, `collect`, `show`, `display`, `write`, `first` |

Spark waits until an action, then optimizes the **whole** plan at once. For example, it pushes filters down to the file scan so less data is read.

In [ ]:
t0 = time.perf_counter()
big_orders = orders.filter(F.col("amount") > 100).withColumn("is_big", F.lit(True))
print(f"Built the plan in {time.perf_counter() - t0:.4f}s. No data has been processed yet.")

Now run an **action**. `display` makes Spark optimize the plan and execute it. This is when the data is actually read and filtered.

In [ ]:
display(big_orders)    # <- the action: the job actually runs now

### DEMO: Peek at the plan Spark built

In [ ]:
big_orders.explain()

**Breakdown:** read it **bottom-up**. The scan (here an in-memory `LocalTableScan`, a file scan on a real table) comes first, then `Filter (amount > 100)`, then `Project`
(the `is_big` column). On a Delta table you'd see `PushedFilters` inside the scan, meaning the filter is applied while reading. You'll read much bigger plans at Mid level.

### DEMO: The everyday API

In [ ]:
display(
    orders
    .filter(F.col("status") == "delivered")                                # WHERE
    .withColumn("amount_with_tax", F.round(F.col("amount") * 1.08, 2))     # derived column
    .select("order_id", "customer_id", "amount", "amount_with_tax")       # SELECT
)

### DEMO: Aggregate with groupBy
`groupBy().agg()` collapses the rows to one per customer, with the number of orders, the total, and the largest order, sorted by total. It's SQL's `GROUP BY`.

In [ ]:
display(
    orders.groupBy("customer_id")                                          # GROUP BY
          .agg(F.count("*").alias("n_orders"),
               F.round(F.sum("amount"), 2).alias("total"),
               F.max("amount").alias("biggest"))
          .orderBy(F.desc("total"))                                        # ORDER BY
)

### CONCEPT: Join types
| `how=` | Returns | SQL equivalent |
|---|---|---|
| `inner` | rows that match on both sides | `JOIN` |
| `left` | every left row; right columns NULL if there's no match | `LEFT JOIN` |
| `left_semi` | left rows that **have** a match (left columns only, never duplicated) | `WHERE EXISTS` |
| `left_anti` | left rows with **no** match | `WHERE NOT EXISTS` |
| `full` | everything from both sides | `FULL OUTER JOIN` |

### KNOWLEDGE CHECK
There are 6 customers and 10 orders. Customers **5 and 6 have no orders**, and customer 1 has 3. Predict the row count of `customers.join(orders, "customer_id", how)` for each
`how`: inner, left, left_semi, left_anti.

In [ ]:
for how in ["inner", "left", "left_semi", "left_anti"]:
    print(f"{how:<10} -> {customers.join(orders, 'customer_id', how).count()} rows")

**Answer and breakdown**
- **inner = 10**: one row per order (each order has a customer).
- **left = 12**: the 10 matches plus customers 5 and 6 with NULL order columns.
- **left_semi = 4**: customers 1-4, **once each**, even though customer 1 has 3 orders. A semi join never multiplies rows.
- **left_anti = 2**: customers 5 and 6.

### CONCEPT: Built-in functions beat Python UDFs
A Python UDF ships **every row** from the JVM to a Python process and back, and the optimizer can't see inside it.
Built-ins (`F.upper`, `F.when`, `F.regexp_replace`, `F.date_add`, ...) run natively and can use Photon.

In [ ]:
from pyspark.sql.functions import udf

@udf("string")
def upper_udf(s):
    return s.upper() if s else None

display(customers.select("name", upper_udf("name").alias("via_udf"), F.upper("name").alias("via_builtin")))

Same result, but the built-in is much faster at scale. Conditional logic doesn't need a UDF either:

In [ ]:
display(orders.select("order_id", "amount",
                      F.when(F.col("amount") >= 150, "high")
                       .when(F.col("amount") >= 50, "medium")
                       .otherwise("low").alias("tier")))

### YOUR TURN (J-6): Total amount per status
Build a DataFrame with columns **`status`** and **`total`** (the sum of `amount`, rounded to 2 decimals), one row per status.

In [ ]:
j6 = None   # TODO: orders.groupBy(...)...

lesson.check("J-6", j6)

# Lesson 7: Data warehousing foundations (your spike)

### CONCEPT: OLTP vs OLAP
| | OLTP (the app database) | OLAP (the warehouse) |
|---|---|---|
| Workload | many tiny reads/writes ("update order 42") | few huge reads ("revenue by region by month") |
| Model | normalized (3NF) | dimensional (star schema) |
| Storage | row-oriented | **columnar** (Parquet/Delta): read only the columns you need |

On Databricks the warehouse is a set of **Delta tables** queried by Spark or Databricks SQL.

### CONCEPT: Facts, dimensions, grain
```
                dim_date
                   │
 dim_customer ── fact_sales ── dim_product        <- a STAR schema: one hop from the fact to every dimension
                   │
                dim_store
```
- **Fact table**: one row per *event* (a sale). It holds **measures** (amount, quantity) and **foreign keys** to dimensions. Long and narrow.
- **Dimension**: the *context* (who, what, where, when). Short and wide.
- **Grain**: the exact meaning of ONE fact row, e.g. "one row per order line". **Declare it first.** Mixing grains causes double counting.

### DEMO: A star-schema query

In [ ]:
orders.createOrReplaceTempView("j_fact_orders")
customers.createOrReplaceTempView("j_dim_customer")

display(spark.sql("""
  SELECT c.region,
         count(*)                AS orders,
         round(sum(o.amount), 2) AS revenue
  FROM j_fact_orders o
  JOIN j_dim_customer c ON o.customer_id = c.customer_id    -- fact -> dimension
  WHERE o.status = 'delivered'
  GROUP BY c.region
  ORDER BY revenue DESC
"""))

**Breakdown:** this is the shape of 80% of BI queries: *filter the fact, join to dimensions, group by dimension attributes, aggregate the measures.*

### CONCEPT: Slowly Changing Dimensions
A customer moves from NA to EMEA. What should *last year's* sales report say?
| Type | What happens | Last year's sales show |
|---|---|---|
| **SCD1** | overwrite the region | EMEA (history is rewritten) |
| **SCD2** | close the old row, insert a new version with `valid_from`/`valid_to`/`is_current` | NA (the truth at the time) |

In [ ]:
from datetime import date
from warehouse.scd import scd1_upsert, scd2_merge

before = [{"customer_id": 1, "region": "NA"}]
change = [{"customer_id": 1, "region": "EMEA"}]

print("SCD1 result:")
for row in scd1_upsert(before, change, key="customer_id"):
    print("  ", row)

Now the same change with **SCD2**: load the customer as NA on 2025-01-01, then apply the EMEA change on 2025-06-01. Watch for **two** rows in the output.

In [ ]:
dim = scd2_merge([], before, key="customer_id", tracked=["region"], as_of=date(2025, 1, 1))
dim = scd2_merge(dim, change, key="customer_id", tracked=["region"], as_of=date(2025, 6, 1))

print("SCD2 result:")
for row in dim:
    print("  ", row)

**Breakdown of the SCD2 output**
- `sk=1`: NA, valid from 2025-01-01 **to** 2025-06-01, `is_current=False`.
- `sk=2`: EMEA, valid from 2025-06-01, `valid_to=None` (still open), `is_current=True`.
- `sk` is the **surrogate key**. Facts point at `sk`, not at `customer_id`, so each sale links to the version that was true at the time.

### KNOWLEDGE CHECK
A sale happened on **2025-03-15**. Which `sk` should it point to?

**Answer:** `sk=1` (NA), because 2025-03-15 falls in `[2025-01-01, 2025-06-01)`. You'll write this "point-in-time" join yourself at Mid level.

### CONCEPT: Medallion architecture
| Layer | Contents | Typical work |
|---|---|---|
| **Bronze** | raw data, as received (duplicates and all) | ingest with Auto Loader, add ingest metadata |
| **Silver** | cleansed, deduplicated, typed, conformed | dedupe, validate, standardize |
| **Gold** | business-ready: star schemas, aggregates | facts, dimensions, KPIs for BI |

### YOUR TURN (J-7): Orders per customer, including customers with none
Columns **`name`** and **`n_orders`**, with **all 6 customers** included (Eli and Fay have `0`).
*Trap:* think carefully about which column you count.

In [ ]:
j7 = None   # TODO

lesson.check("J-7", j7)

# Class summary
| Lesson | Key idea | Complexity |
|---|---|---|
| 1. Method | clarify, brute force, optimize, trace, complexity | |
| 2. Hash maps | `in set` / dict lookups are O(1); counting replaces sorting | O(n) |
| 3. One pass | carry running state; two pointers from both ends | O(n), O(1) space |
| 4. Binary search | sorted input; keep the invariant; `mid ± 1` | O(log n) |
| 5. BFS | queue + visited set; fewest hops | O(V + E) |
| 6. PySpark | lazy plan, actions run it; join types; built-ins over UDFs | |
| 7. Warehousing | fact/dimension/grain; SCD1 vs SCD2; bronze/silver/gold | |

### Before the test, check yourself
- [ ] All 7 **YOUR TURN** cells show `PASS`
- [ ] You can explain *why* each knowledge-check answer is what it is

**Next: open `11_junior_test`** (~60 minutes, pass mark 85%).